In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost lightgbm tqdm -q

clear_output()
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
#Read the dataset Q1_data.csv using read_csv()
df = pd.read_csv(f"{path}/Q1_data.csv")



In [ ]:
# Task 2: Write your code here:
#Inspect the first few rows using head()
df.head()


In [ ]:
# Task 3: Write your code here:
#Display dataset information using info()
df.info()


In [ ]:
# Task 4: Write your code here:
#Show statistical description using describe()

df.describe()

In [ ]:
# Task 5: Write your code here:
#Plot the target distribution (delivery_time)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery_time Distribution')
plt.xlabel('delivery_time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
#Drop the 'Order_ID' column from the data
df_clean = df.copy()
df_clean.drop('Order_ID',axis=1, inplace=True)

In [ ]:
# Task 2: Write your code here:
#Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )

# Fill with mode (categorical, few missing)
df_clean['Weather'].fillna(df_clean['Weather'].mode()[0], inplace=True)
df_clean['Traffic_Level'].fillna(df_clean['Traffic_Level'].mode()[0], inplace=True)
df_clean['Time_of_Day'].fillna(df_clean['Time_of_Day'].mode()[0], inplace=True)

#Fill with median (numerical, sensitive to outliers)
df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].median(), inplace=True)
df_clean['Delivery_Time'].fillna(df_clean['Delivery_Time'].median(), inplace=True)

print("Missing values after cleaning:")
print(df_clean.isnull().sum())
print("\nShape after dropping Cabin:", df_clean.shape)

In [ ]:
# Task 3: Write your code here:
#Check and remove duplicates if any exist
duplicates = df_clean.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

if duplicates > 0:
    df_clean.drop_duplicates(inplace=True)
    print(f"Removed {duplicates} duplicate rows")
else:
    print("No duplicates found!")

In [ ]:
# Task 4: Write your code here:
#Encode categorical variables if needed (Bonus if used One Hot Encoding)

from sklearn.preprocessing import LabelEncoder #import OneHotEncoder

for col in ['Weather','Traffic_Level', 'Time_of_Day', 'Vehicle_Type']:

    oe = LabelEncoder() # Instantiate OneHotEncoder
    df_clean[col] = oe.fit_transform(df_clean[col])

df_clean.head()

In [ ]:
# Task 5: Write your code here:
#Apply feature scaling for all features (Use StandardScaler)

from sklearn.preprocessing import StandardScaler
features_to_scale=['Distance_km','Weather','Traffic_Level','Time_of_Day','Vehicle_Type','Preparation_Time_min','Courier_Experience_yrs']
standard_scaler = StandardScaler() # Instantiate StandardScaler
data_standard=df_clean.copy()
data_standard[features_to_scale] = standard_scaler.fit_transform(df_clean[features_to_scale]) # Apply fit_transform

print('\nData after scaling:\n', data_standard) #show after scaling

In [ ]:
# Task 6: Write your code here:
#Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)
# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()
  plt.show()

check_target_imbalance(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
#Split the dataset into features (X) and target (y)
X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
#2
#from sklearn.model_selection import KFold
#Use the correct split: KFold OR StratifiedKFold
#from sklearn.model_selection import train_test_split, KFold
#kfold = KFold(n_splits=5, shuffle=True, random_state=42)
#for train_idx, val_idx in kfold.split(X_train_scaled):
  #  X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
  #  y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    #model.fit(X_fold_train, y_fold_train)
    #y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    #mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    #rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))



#Train a RandomForest model
from sklearn.ensemble import RandomForestRegressor

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Scale features - fit on train, transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
print(f"RMSE: ${rmse_scores.mean():,.2f}")
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
print("Model trained!")
#Evaluate using MAE (Mean Absolute Error) ONLY
#Print the averaged score across all folds

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: